# Rancher Kubernetes (RKE2) on FABRIC via Ansible

This notebook provisions a **2-node** FABRIC slice on a single site and uses Ansible to deploy [RKE2](https://docs.rke2.io/) (Rancher Kubernetes Engine 2).

| Role | Node | Resources |
|------|------|-----------|
| Control plane (`rke2-server`) | `node1` | 8 cores, 16 GB RAM |
| Worker (`rke2-agent`) | `node2` | 8 cores, 16 GB RAM |

Cluster traffic uses a private L2 dataplane (`192.168.1.0/24`). Ansible reaches the nodes over the FABRIC management network.

This is an updated notebook that will continuously test the sites to find a suitable one to launch. 

## Import the FABlib Library

In [1]:
import random
import time
import os
import subprocess

from ipaddress import IPv4Network
from pathlib import Path

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.verify_and_configure()

# Find a Site With Enough Capacity

resources = fablib.get_resources()
resources.update()
siteList = resources.get_site_names()

nodesReq,coresReq,ramReq = 2,8,10

# Scale up quite a bit to ensure there are plenty of resources available.
totalCoreAvail = nodesReq * coresReq * 2
totalRamAvail = nodesReq * ramReq * 2

for siteName in siteList:
    cores = resources.get_core_available(siteName)
    ram = resources.get_ram_available(siteName)
    if cores < totalCoreAvail or ram < totalRamAvail:
        print(f"{siteName} does not have enough free cores/RAM")
        continue
    sliceName = "rke2-" + siteName
    network_name = "rke2net"

    # Clean up left-over slices with the same names
    existing = None
    for s in fablib.get_slices():
        if s.get_name() == sliceName:
            print(f"Deleting existing slice: {sliceName}")
            s.delete()
            break

    # Set up new slice
    slice = fablib.new_slice(name=sliceName)
    net = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))
    
    for i in range(1, nodesReq + 1):
        node = slice.add_node(name=f"node{i}",
                              site=siteName,
                              cores=coresReq,ram=ramReq,disk=50,
                              image="default_ubuntu_22",)
        iface = node.add_component(model="NIC_Basic", name="nic").get_interfaces()[0]
        iface.set_mode("config")
        net.add_interface(iface)

    print("==========================================================================")
    try:
        print(f"Submitting slice at {siteName}...")
        slice.submit(progress=False)
    except Exception as e:
        print(f"{siteName}: slice submission failed with error {e}")
        try:
            slice.delete()
        except Exception:
            pass
        continue

    while True:
        time.sleep(10)
        slice.update()
        slice_state = slice.get_state()
        print(f"Slice state: {slice_state}")
        if slice_state == "Closing":
            print(f"Need to find new site")
            break
        else: 
            print("Slice stable:", slice.isStable())
        nodes = slice.get_nodes()
        if all(node.get_management_ip() is not None for node in nodes):
            for node in nodes:
                print("----", node.get_name(), "----")
                print("management ip:", node.get_management_ip())
                print(node.get_ssh_command())
            break
            
    # Setup networking
    for i in range(nodesReq):
        node = slice.get_node(name=f"node{i + 1}")
        iface = node.get_interface(network_name=network_name)
        iface.ip_link_up()
        iface.ip_addr_add(
            addr=f"192.168.1.{i + 1}",
            subnet=IPv4Network("192.168.1.0/24"),
        )
        print(f"{node.get_name()} -> 192.168.1.{i + 1}")
    
    # Ansible preparation
    node_defs = []
    for node in slice.get_nodes():
        name = node.get_name()
        private_ip = str(node.get_interface(network_name=network_name).get_ip_addr())
        group = "rke2_servers" if name == "node1" else "rke2_agents"
        node_defs.append({"name": name, "private_ip": private_ip, "group": group})
    
    slice_key = fablib.get_default_slice_key()["slice_private_key_file"]
    ssh_config = "/home/fabric/work/fabric_config/ssh_config"
    
    for nd in node_defs:
        node = slice.get_node(nd["name"])
        stdout, stderr = node.execute(
            "python3 -c 'import sys; print(sys.executable)'",
            quiet=True,
        )
        nd["node"] = node
        nd["python"] = stdout.strip()
    
    lines = []
    lines.append("all:")
    lines.append("  vars:")
    lines.append("    ansible_become: true")
    lines.append(f'    ansible_ssh_private_key_file: "{slice_key}"')
    lines.append(f'    ansible_ssh_common_args: "-F {ssh_config}"')
    lines.append('    rke2_token: "fabric-rke2-cluster-token"')
    lines.append('    rke2_server_private_ip: "192.168.1.1"')
    lines.append("")
    lines.append("  children:")
    
    for group in ("rke2_servers", "rke2_agents"):
        lines.append(f"    {group}:")
        lines.append("      hosts:")
        for nd in node_defs:
            if nd["group"] != group:
                continue
            node = nd["node"]
            lines.append(f'        {nd["name"]}:')
            lines.append(f'          ansible_host: "{node.get_management_ip()}"')
            lines.append(f'          ansible_user: "{node.get_username()}"')
            lines.append(f'          ansible_python_interpreter: "{nd["python"]}"')
            lines.append(f'          private_ip: "{nd["private_ip"]}"')
        lines.append("")
    
    inventory = "\n".join(lines) + "\n"
    Path("playbook").mkdir(exist_ok=True)
    Path("playbook/inventory.yml").write_text(inventory)
    print("Wrote playbook/inventory.yml")

    # Playbook: Host Prerequisites

    Path("logs").mkdir(exist_ok=True)
    log_path = Path("logs") / f"{siteName}-ansible.log"

    with open(log_path, "w") as log_file:
        print("==== Running prerequisite playbook ====")
        prereq_result = subprocess.run(["ansible-playbook","-i","playbook/inventory.yml","playbook/playbook-prereqs.yml",],
            stdout=log_file,stderr=subprocess.STDOUT,text=True,
        )
    
        if prereq_result.returncode != 0:
            print(f"{siteName}: prerequisite playbook failed")
            slice.delete()
            continue
    
        print("==== Running RKE2 playbook ====")
        rke2_result = subprocess.run(["ansible-playbook","-i","playbook/inventory.yml","playbook/playbook-rke2.yml",],
            stdout=log_file,stderr=subprocess.STDOUT,text=True,
        )
    
        if rke2_result.returncode != 0:
            print(f"{siteName}: RKE2 playbook failed")
            slice.delete()
            continue

    # RKE2 Validation

    server = slice.get_node("node1")
    print("==== nodes ====")
    stdout, stderr = server.execute("kubectl get nodes -o wide", quiet=True)
    print(stdout)
    
    print("==== nginx-demo ====")
    stdout, stderr = server.execute("kubectl get pods,svc -l app=nginx-demo -o wide", quiet=True)
    print(stdout)
    
    print("==== curl via NodePort on dataplane ====")
    stdout, stderr = server.execute("curl -s -o /dev/null -w '%{http_code}\n' http://192.168.1.1:30080/", quiet=True)
    if stdout.strip() == "200":
        print(f"{siteName} works ...")
        break


User: LY996437@wcupa.edu bastion key is valid!
Configuration is valid
User: LY996437@wcupa.edu bastion key is valid!
Configuration is valid
Please save the config!
STAR does not have enough free cores/RAM
Submitting slice at KANS...
Running post boot config threads ...
Post boot config node1, Done! (1 sec)
Post boot config node2, Done! (1 sec)
Saving fablib data...  Done!
Slice state: StableOK
Slice stable: True
---- node1 ----
management ip: 2001:400:a100:3060:f816:3eff:fe0c:276f
ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3060:f816:3eff:fe0c:276f
---- node2 ----
management ip: 2001:400:a100:3060:f816:3eff:fe63:9030
ssh -i /home/fabric/.ssh/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:400:a100:3060:f816:3eff:fe63:9030
node1 -> 192.168.1.1
node2 -> 192.168.1.2
Wrote playbook/inventory.yml
==== Running prerequisite playbook ====
==== Running RKE2 playbook ====
==== nodes ====
NAME    STATUS   ROLES         

In [22]:
slice.delete()